In [3]:
import pandas as pd
import requests 
from PIL import Image
from io import BytesIO



# Now load the training dataset


In [4]:
train_df = pd.read_csv("data/processed/train_20k.csv")

print("Dataset shape:", train_df.shape)
print("Available image URLs:", train_df["image_url"].notna().sum())

Dataset shape: (20000, 7)
Available image URLs: 20000


# test 20 images URLS

In [5]:
success = 0
failed = 0

test_urls = train_df["image_url"].head(20)

for i, url in enumerate(test_urls):

    try:
        response = requests.get(
            url,
            timeout=15,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        if response.status_code == 200:

            img = Image.open(BytesIO(response.content))
            img.verify()

            success += 1
            print(f"{i+1}. SUCCESS")

        else:
            failed += 1
            print(f"{i+1}. FAILED - HTTP {response.status_code}")

    except Exception as e:
        failed += 1
        print(f"{i+1}. FAILED - {type(e).__name__}")

print("\n========== FINAL RESULT ==========")
print("Successful images:", success)
print("Failed images:", failed)
print("Total tested:", success + failed)

1. SUCCESS
2. SUCCESS
3. SUCCESS
4. SUCCESS
5. SUCCESS
6. SUCCESS
7. SUCCESS
8. SUCCESS
9. SUCCESS
10. SUCCESS
11. SUCCESS
12. SUCCESS
13. SUCCESS
14. SUCCESS
15. SUCCESS
16. SUCCESS
17. SUCCESS
18. SUCCESS
19. SUCCESS
20. SUCCESS

========== FINAL RESULT ==========
Successful images: 20
Failed images: 0
Total tested: 20


# now create the image folder

In [6]:
import os 

folders = [
    "../images/train",
    "../images/validation",
    "../images/test"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Image folders created successfully!")

Image folders created successfully!


# download 2000 training images

In [7]:
import pandas as pd
import requests
from PIL import Image, UnidentifiedImageError
from io import BytesIO
import os
import time

train_df = pd.read_csv("data/processed/train_20k.csv")

download_dir = "../images/train"
os.makedirs(download_dir, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

successful = []
failed = []

for i, row in train_df.head(2000).iterrows():

    image_id = str(row["id"])
    image_url = row["image_url"]

    image_path = os.path.join(download_dir, image_id + ".jpg")

    if os.path.exists(image_path):
        successful.append(image_id)
        continue

    downloaded = False

    for attempt in range(3):

        try:
            response = session.get(image_url, timeout=20)
            response.raise_for_status()

            img = Image.open(
                BytesIO(response.content)
            )

            img.load()
            img = img.convert("RGBA").convert("RGB")

            img.save(image_path, format="JPEG")

            successful.append(image_id)
            downloaded = True
            break

        except Exception as e:
            error = type(e).__name__
            time.sleep(1)

    if not downloaded:
        failed.append({
            "id": image_id,
            "image_url": image_url,
            "error": error
        })

    if (i + 1) % 100 == 0:
        print(
            f"Processed: {i+1}/2000 | "
            f"Success: {len(successful)} | "
            f"Failed: {len(failed)}"
        )

# Save download records
pd.DataFrame({"id": successful}).to_csv(
    "../images/successful_images.csv", index=False
)

pd.DataFrame(failed).to_csv(
    "../images/failed_images.csv", index=False
)

print("\n========== FINAL RESULTS ==========")
print("Successful:", len(successful))
print("Failed:", len(failed))
print("Total:", len(successful) + len(failed))

KeyboardInterrupt: 

# Prepare the Image dataset

In [ ]:
#laod the original dataset
train_df = pd.read_csv("data/processed/train_20k.csv")

#load only downloaded imaage 

success_df = pd.read_csv("../images/successful_images.csv")

#keep only successful downloaded image

image_df = train_df.merge(
    success_df,
    on = "id",
    how = "inner"
)

# create the local image path 

image_df["image_path"] = image_df["id"].apply(
    lambda x: os.path.join(
        "../images/train",
        str(x) + ".jpg"
    )
)

#keep required columns

image_df = image_df[
    ["id" , "clean_title" , "image_path" , "2_way_label"] 
]

# Verify image paths
image_df = image_df[
    image_df["image_path"].apply(os.path.exists)
].copy()

# Save processed image dataset
image_df.to_csv(
    "data/processed/image_train_1889.csv",
    index=False
)

print("Image dataset prepared!")
print("Total images:", len(image_df))
print("\nLabel distribution:")
print(image_df["2_way_label"].value_counts())


In [ ]:
# now train the model 

from sklearn.model_selection import train_test_split

image_train_df , image_val_df = train_test_split(
    image_df,
    test_size = 0.2,
    random_state = 42,
    stratify=image_df["2_way_label"]             
)

print("Training images" , len(image_train_df))
print("Validation images:", len(image_val_df))

print("\nTraining labels:")
print(image_train_df["2_way_label"].value_counts())

print("\nValidation labels:")
print(image_val_df["2_way_label"].value_counts())

# open AI pretrained CLIP model  

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

CLIP_NAME = "openai/clip-vit-base-patch32"

clip_model = CLIPModel.from_pretrained(CLIP_NAME)

clip_processor = CLIPProcessor.from_pretrained(CLIP_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

clip_model = clip_model.to(device)

clip_model.eval()

print("CLIP loaded successfully!")
print("Device:", device)

In [ ]:
# Freeze clip parameters

for params in clip_model.parameters():
    params.requires_grad = False

clip_model.eval()

print("CLIP parameters frozen successfully!")


# Extract image embeddings

In [ ]:
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class ImageDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        inputs = clip_processor(
            images=image,
            return_tensors="pt"
        )

        return {
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "label": int(row["2_way_label"]),
            "id": str(row["id"])
        }


image_dataset = ImageDataset(image_train_df)

image_loader = DataLoader(
    image_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Image DataLoader created!")
print("Images:", len(image_dataset))

# Extract clip freatures

In [ ]:
with torch.no_grad():
    sample = next(iter(image_loader))
    pixel_values = sample["pixel_values"].to(device)

    output = clip_model.get_image_features(
        pixel_values=pixel_values
    )

print(type(output))
print(output)

In [ ]:
all_features = []
all_labels = []
all_ids = []

clip_model.eval()

with torch.no_grad():

    for batch in tqdm(image_loader):

        pixel_values = batch["pixel_values"].to(device)

        # Extract image features
        vision_output = clip_model.vision_model(
            pixel_values=pixel_values
        )

        # Apply CLIP visual projection
        features = clip_model.visual_projection(
            vision_output.pooler_output
        )

        # Normalize embeddings
        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        all_features.append(features.cpu().numpy())
        all_labels.extend(batch["label"].numpy())
        all_ids.extend(batch["id"])

        del pixel_values, vision_output, features

        torch.cuda.empty_cache()

image_features = np.concatenate(all_features, axis=0)
image_labels = np.array(all_labels)

print("Feature extraction completed!")
print("Feature shape:", image_features.shape)
print("Labels shape:", image_labels.shape)

In [ ]:
# save the embeddings and extracted features

import os 
import numpy as np 

os.makedirs("../features" , exist_ok = True)

np.save("../features/clip_train_features.npy" , image_features)
np.save("../features/clip_train_labels.npy" , image_labels)

print("Clip features saved successfully")

In [ ]:
# train an image classifier 

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score , precision_score , f1_score , recall_score , classification_report , confusion_matrix)

clip_classifier = LogisticRegression(
    max_iter = 2000,
    random_state = 42
)

clip_classifier.fit(image_features , image_labels)
print("CLIP image classifier trained successfully!")

In [ ]:
#extract validation feature 

val_image_dataset = ImageDataset(image_val_df)

val_image_loader = DataLoader(
    val_image_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

val_features = []
val_labels = []

clip_model.eval()

with torch.no_grad():
    for batch in tqdm(val_image_loader):

        pixel_values = batch["pixel_values"].to(device)

        vision_output = clip_model.vision_model(
            pixel_values=pixel_values
        )

        features = clip_model.visual_projection(
            vision_output.pooler_output
        )

        features = features / features.norm(
            dim=-1,
            keepdim=True
        )

        val_features.append(features.cpu().numpy())
        val_labels.extend(batch["label"].numpy())

        del pixel_values, vision_output, features
        torch.cuda.empty_cache()

val_features = np.concatenate(val_features, axis=0)
val_labels = np.array(val_labels)

print("Validation features:", val_features.shape)

In [ ]:
clip_predictions = clip_classifier.predict(val_features)

print("CLIP Image Model Results")
print("------------------------")

print("Accuracy:", accuracy_score(val_labels, clip_predictions))
print("Precision:", precision_score(val_labels, clip_predictions))
print("Recall:", recall_score(val_labels, clip_predictions))
print("F1 Score:", f1_score(val_labels, clip_predictions))

print("\nClassification Report:")
print(classification_report(val_labels, clip_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(val_labels, clip_predictions))

In [ ]:
# save the clip result

import json
import os

os.makedirs("../results", exist_ok=True)

clip_metrics = {
    "model": "CLIP",
    "accuracy": float(accuracy_score(val_labels, clip_predictions)),
    "precision": float(precision_score(val_labels, clip_predictions)),
    "recall": float(recall_score(val_labels, clip_predictions)),
    "f1_score": float(f1_score(val_labels, clip_predictions)),
    "training_images": len(image_train_df),
    "validation_images": len(image_val_df)
}

with open("../results/clip_results.json", "w") as f:
    json.dump(clip_metrics, f, indent=4)

print("CLIP results saved successfully!")

In [ ]:
#save the image embeddings

import numpy as np

np.save("../features/clip_train_features.npy", image_features)
np.save("../features/clip_train_labels.npy", image_labels)

np.save("../features/clip_val_features.npy", val_features)
np.save("../features/clip_val_labels.npy", val_labels)

print("All CLIP features saved!")